In [ ]:
import os
from pathlib import Path

nb_dir = Path.cwd()
        

target = (nb_dir / '..' / '..').resolve()
os.chdir(target)




In [ ]:
from grasp.graph.graph_storage import GraphStorage, load_graph_storage

gs_path = "data/clearscope_e3/graph_storage/clearscope_e3_default_experiment_dataset-clearscope_e3_context_size-120_step_size-120_graph_storage.pt"
gs: GraphStorage = load_graph_storage(gs_path)


print(gs)
known_executables = gs.train_subject_cmds
print(len(known_executables))
print(len(known_executables))

608
608


In [3]:
gs.train_subject_cmds

['system_server',
 'com.android.systemui',
 'com.android.settings:CryptKeeper',
 'system_server',
 'com.android.systemui',
 'com.android.calendar',
 'com.android.browser',
 'com.android.settings:CryptKeeper',
 'system_server',
 'com.android.systemui',
 'com.android.settings:CryptKeeper',
 'system_server',
 'com.android.systemui',
 'com.android.calendar',
 'com.android.settings:CryptKeeper',
 'system_server',
 'com.android.systemui',
 'com.android.inputmethod.latin',
 'com.android.settings:CryptKeeper',
 'system_server',
 'com.android.systemui',
 'com.android.settings:CryptKeeper',
 'system_server',
 'com.android.systemui',
 'com.android.calendar',
 'com.android.calendar',
 'com.android.browser',
 'com.android.settings:CryptKeeper',
 'system_server',
 'com.android.systemui',
 'com.android.settings:CryptKeeper',
 'system_server',
 'com.android.systemui',
 'com.android.settings:CryptKeeper',
 'system_server',
 'com.android.systemui',
 'com.android.calendar',
 'com.android.settings:CryptKe

In [4]:
gs.train_subject_cmd_to_id

{'android.process.acore': 0,
 'android.process.media': 1,
 'com.android.browser': 2,
 'com.android.calculator2': 3,
 'com.android.calendar': 4,
 'com.android.camera2': 5,
 'com.android.contacts': 6,
 'com.android.defcontainer': 7,
 'com.android.deskclock': 8,
 'com.android.dialer': 9,
 'com.android.documentsui': 10,
 'com.android.email': 11,
 'com.android.externalstorage': 12,
 'com.android.gallery3d': 13,
 'com.android.inputmethod.latin': 14,
 'com.android.launcher3': 15,
 'com.android.messaging': 16,
 'com.android.music': 17,
 'com.android.musicfx': 18,
 'com.android.nfc': 19,
 'com.android.packageinstaller': 20,
 'com.android.phone': 21,
 'com.android.providers.calendar': 22,
 'com.android.quicksearchbox': 23,
 'com.android.settings': 24,
 'com.android.settings:CryptKeeper': 25,
 'com.android.systemui': 26,
 'com.motorola.android.buacontactadapter': 27,
 'com.svox.pico': 28,
 'org.mozilla.fennec_firefox_dev': 29,
 'system_server': 30}

In [5]:
len(set(known_executables))

31

In [6]:
known_executables_set = set(known_executables)
known_executables_list = sorted(known_executables_set)
print(len(known_executables_list))
known_executables_list


31


['android.process.acore',
 'android.process.media',
 'com.android.browser',
 'com.android.calculator2',
 'com.android.calendar',
 'com.android.camera2',
 'com.android.contacts',
 'com.android.defcontainer',
 'com.android.deskclock',
 'com.android.dialer',
 'com.android.documentsui',
 'com.android.email',
 'com.android.externalstorage',
 'com.android.gallery3d',
 'com.android.inputmethod.latin',
 'com.android.launcher3',
 'com.android.messaging',
 'com.android.music',
 'com.android.musicfx',
 'com.android.nfc',
 'com.android.packageinstaller',
 'com.android.phone',
 'com.android.providers.calendar',
 'com.android.quicksearchbox',
 'com.android.settings',
 'com.android.settings:CryptKeeper',
 'com.android.systemui',
 'com.motorola.android.buacontactadapter',
 'com.svox.pico',
 'org.mozilla.fennec_firefox_dev',
 'system_server']

In [ ]:
import time
from urllib.parse import unquote, urlparse

import psycopg2  # type: ignore
from psycopg2 import sql  # type: ignore

from grasp import config
from grasp.schema import DatasetName

user = "postgres"
password = "lolroflomg"
host = config.DB_HOST
port = 9889

base_url = f"postgresql://{user}:{password}@{host}:{port}"
connection_real_data = f"{base_url}/{DatasetName.CLEARSCOPE_E3.value}"

# Known executable commands from training graph storage
known_executables_set = set(known_executables)
known_executables_list = sorted(known_executables_set)

new_table_name = "subject_node_table"
backup_table_name = f"{new_table_name}_backup"

# Parse connection URL once
parsed = urlparse(connection_real_data)
dbname = parsed.path.lstrip('/') if parsed.path else None
db_user = unquote(parsed.username) if parsed.username else None
db_password = unquote(parsed.password) if parsed.password else None
db_host = parsed.hostname
db_port = parsed.port

t0 = time.perf_counter()
conn = psycopg2.connect(
    dbname=dbname,
    user=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
    application_name="remove_process_unknown_exec_fast",
)

try:
    with conn, conn.cursor() as cur:
        # Fast path: keep one immutable backup, recreate filtered table from it.
        # If backup already exists, reuse it. If not, rename original once.
        cur.execute("SELECT to_regclass(%s)", (new_table_name,))
        has_main = cur.fetchone()[0] is not None # type: ignore
        cur.execute("SELECT to_regclass(%s)", (backup_table_name,))
        has_backup = cur.fetchone()[0] is not None # type: ignore

        if not has_main and not has_backup:
            raise RuntimeError(
                f"Neither '{new_table_name}' nor '{backup_table_name}' exists."
            )

        if has_main and not has_backup:
            t_rename = time.perf_counter()
            cur.execute(
                sql.SQL("ALTER TABLE {} RENAME TO {}").format(
                    sql.Identifier(new_table_name),
                    sql.Identifier(backup_table_name),
                )
            )
            print(
                f"Renamed original table to backup in "
                f"{time.perf_counter() - t_rename:.3f}s"
            )
        elif has_main and has_backup:
            # Keep existing backup as source of truth; refresh working table below.
            print(
                f"Both '{new_table_name}' and '{backup_table_name}' exist; "
                "keeping backup and refreshing working table."
            )

        source_table = backup_table_name if has_backup or has_main else new_table_name

        # Remember node_uuids that will be removed (unknown execs + NULL cmd)
        t_collect = time.perf_counter()
        cur.execute(
            sql.SQL(
                """
                    SELECT hash_id
                    FROM {}
                    WHERE cmd IS NULL OR NOT (cmd = ANY(%s))
                    """
            ).format(sql.Identifier(source_table)),
            (known_executables_list,),
        )
        deleted_node_hash_ids = [row[0] for row in cur.fetchall()]
        print(
            f"Collected {len(deleted_node_hash_ids)} deleted node_hash_ids in "
            f"{time.perf_counter() - t_collect:.3f}s"
        )

        t_rebuild = time.perf_counter()
        cur.execute(
            sql.SQL("DROP TABLE IF EXISTS {}").format(
                sql.Identifier(new_table_name)
            )
        )
        cur.execute(
            sql.SQL("CREATE TABLE {} (LIKE {} INCLUDING ALL)").format(
                sql.Identifier(new_table_name),
                sql.Identifier(source_table),
            )
        )
        cur.execute(
            sql.SQL(
                "INSERT INTO {} SELECT * FROM {} WHERE cmd = ANY(%s)"
            ).format(
                sql.Identifier(new_table_name),
                sql.Identifier(source_table),
            ),
            (known_executables_list,),
        )

        inserted_rows = cur.rowcount
        print(
            f"Rebuilt filtered '{new_table_name}' with {inserted_rows} rows in "
            f"{time.perf_counter() - t_rebuild:.3f}s"
        )

        # Useful sanity numbers
        cur.execute(
            sql.SQL("SELECT COUNT(*) FROM {}").format(
                sql.Identifier(source_table)
            )
        )
        source_count = cur.fetchone()[0] # type: ignore
        print(f"Source rows: {source_count}")
        print(f"Deleted rows: {source_count - inserted_rows}")

finally:
    conn.close()

print(f"Total elapsed: {time.perf_counter() - t0:.3f}s")

Both 'subject_node_table' and 'subject_node_table_backup' exist; keeping backup and refreshing working table.
Collected 1569 deleted node_hash_ids in 0.004s
Rebuilt filtered 'subject_node_table' with 10900 rows in 0.133s
Source rows: 12469
Deleted rows: 1569
Total elapsed: 0.156s


In [8]:
deleted_node_hash_ids

['...',
 ...]

In [ ]:
event_table_name = "event_table"

# Deduplicate once for stable/efficient ANY() checks
deleted_node_hash_ids_list = sorted(set(deleted_node_hash_ids))

# Process in batches to reduce memory pressure
BATCH_SIZE = 10000
deleted_batches = [
    deleted_node_hash_ids_list[i : i + BATCH_SIZE]
    for i in range(0, len(deleted_node_hash_ids_list), BATCH_SIZE)
]

t0_event = time.perf_counter()
conn_event = psycopg2.connect(
    dbname=dbname,
    user=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
    application_name="remove_events_with_deleted_nodes_main_only",
)

try:
    with conn_event, conn_event.cursor() as cur_event:
        # Ensure main table exists
        cur_event.execute("SELECT to_regclass(%s)", (event_table_name,))
        has_event_main = cur_event.fetchone()[0] is not None # type: ignore
        if not has_event_main:
            raise RuntimeError(f"Table '{event_table_name}' does not exist.")

        # Count source rows
        cur_event.execute(
            sql.SQL("SELECT COUNT(*) FROM {}").format(sql.Identifier(event_table_name))
        )
        event_source_count = cur_event.fetchone()[0] # type: ignore

        # Count and delete rows in batches
        t_event_count = time.perf_counter()
        total_event_rows_to_delete = 0
        total_event_deleted_rows = 0

        for batch_idx, batch in enumerate(deleted_batches):
            try:
                cur_event.execute(
                    sql.SQL(
                        """
                            SELECT COUNT(*)
                            FROM {}
                            WHERE (src_node IS NOT NULL AND src_node = ANY(%s))
                               OR (dst_node IS NOT NULL AND dst_node = ANY(%s))
                            """
                    ).format(sql.Identifier(event_table_name)),
                    (batch, batch),
                )
                batch_rows_to_delete = cur_event.fetchone()[0] # type: ignore
                total_event_rows_to_delete += batch_rows_to_delete

                cur_event.execute(
                    sql.SQL(
                        """
                            DELETE FROM {}
                            WHERE (src_node IS NOT NULL AND src_node = ANY(%s))
                               OR (dst_node IS NOT NULL AND dst_node = ANY(%s))
                            """
                    ).format(sql.Identifier(event_table_name)),
                    (batch, batch),
                )
                total_event_deleted_rows += cur_event.rowcount
                conn_event.commit()

                print(
                    f"Batch {batch_idx + 1}/{len(deleted_batches)}: "
                    f"Deleted {cur_event.rowcount} rows "
                    f"(counted {batch_rows_to_delete} to delete) "
                    f"in {time.perf_counter() - t_event_count:.3f}s"
                )

            except psycopg2.errors.DiskFull:
                print(f"Disk full error in batch {batch_idx}. Stopping gracefully.")
                conn_event.rollback()
                raise

        print(
            f"Rows to delete from '{event_table_name}': {total_event_rows_to_delete} "
            f"(counted in {time.perf_counter() - t_event_count:.3f}s)"
        )
        print(f"Deleted {total_event_deleted_rows} rows from '{event_table_name}'")
        print(f"Event source rows: {event_source_count}")
        print(f"Event remaining rows: {event_source_count - total_event_deleted_rows}")

finally:
    conn_event.close()

print(f"Total event-table elapsed: {time.perf_counter() - t0_event:.3f}s")


Batch 1/1: Deleted 80548 rows (counted 80548 to delete) in 4.899s
Rows to delete from 'event_table': 80548 (counted in 4.899s)
Deleted 80548 rows from 'event_table'
Event source rows: 18316129
Event remaining rows: 18235581
Total event-table elapsed: 5.614s
